# Keras Functional API の概要

**学習目標**
1.  Sequential API と Functional API の違いを理解する。
2.  Keras Functional API と前処理レイヤーを使用して Wide & Deep モデルを構築する。
3.  `keras.utils.plot_model` を使用してモデルのアーキテクチャを可視化する。

このノートブックでは、Functional API と前処理レイヤーを使用して Wide & Deep モデルを構築します。

### Wide & Deep モデル
Wide & Deep モデルの根底にある考え方は、記憶（memorization）と汎化（generalization）という 2 つの学習方法を組み合わせるため、ワイドな線形モデルとディープ ラーニング モデルの両方を組み込むことです。元の研究論文はこちらで確認できます: [Wide & Deep Learning for Recommender Systems](https://arxiv.org/abs/1606.07792)

<img src='assets/wide_deep.png' width='80%'>
<sup>(画像出典: https://ai.googleblog.com/2016/06/wide-deep-learning-better-together-with.html)</sup>

モデルのワイド（Wide）部分は記憶（memory）要素に関連しています。この場合、交差特徴量（crossed features）の幅広いセットを持つ線形モデルをトレーニングし、これらの関連データと割り当てられたラベルとの相関関係を学習します。モデルのディープ（Deep）部分は汎化（generalization）要素に関連しており、特徴量の埋め込みベクトル（embedding vectors）を使用します。最適な埋め込みはトレーニング プロセスを通じて学習されます。これら 2 つの方法は単独でも適切に機能しますが、Wide & Deep モデルはこれらの技術を組み合わせることで優れた成果を発揮します。

## セットアップ
まず、このラボに必要なライブラリをインポートします。

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import datetime
import shutil

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import Model
from keras.callbacks import TensorBoard
from keras.layers import (
    CategoryEncoding,
    Concatenate,
    Dense,
    Discretization,
    Embedding,
    Flatten,
    HashedCrossing,
    Input,
)
from keras.metrics import RootMeanSquaredError
from matplotlib import pyplot as plt

In [ ]:
%matplotlib inline

### 生データの読み込み

この一連の最初のノートブックで作成した CSV ファイルを使用して、taxifare（タクシー運賃）データセットを利用します。これらのファイルは `../data` に保存されています。

In [ ]:
!ls -l ../data/*.csv

### tf.data を使用した CSV ファイルの読み込み

上記の CSV ファイルからデータを読み込むためのこれらの関数は、[前のノートブック](2_dataset_api.ipynb) で作成したものです。

`tf.data` API は、データを効率的に読み込み、前処理を行います。 
- `parse_csv`: CSV の行を特徴量とラベルにパースします。複数の入力に対する Functional API との互換性のため、特徴量はタプルとして返されます。
- `create_dataset`: `parse_csv` のマッピング、リピート、シャッフル（トレーニング用）、バッチ処理を含め、CSV ファイルから `tf.data.Dataset` を構築します。

In [ ]:
def parse_csv(row):
    ds = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(ds[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    feature = tf.strings.to_number(ds[2:6])  # 一部の特徴量のみを使用
    # それぞれを個別に処理できるように、特徴量をタプルで渡します。
    return (feature[0], feature[1], feature[2], feature[3]), label


def create_dataset(pattern, batch_size, mode="eval"):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(parse_csv).repeat()
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

## Keras Functional API を使用した Wide & Deep モデルの構築

Wide & Deep モデルを構築します。このモデルには次の 2 つのパートがあります。
1.  **Wide パス:** 特定の特徴量の相互作用を記憶するため（多くの場合、カテゴリカル特徴量/交差特徴量を使用）。
2.  **Deep パス:** パターンを汎化するため（多くの場合、DNN を介した数値特徴量/埋め込みを使用）。
その後、予測のために出力が結合されます。Functional API は、この複雑な構造を処理できます。

### 入力レイヤーの定義

Functional API を使用すると、各入力特徴量に独自の [`keras.Input` レイヤー](https://keras.io/api/layers/core_layers/input/) が割り当てられます。これらはデータのエントリ ポイントを定義します。
4 つの座標特徴量に対して `Input` オブジェクトを作成し、`name`、`shape`（スカラー）、`dtype` を指定します。

In [ ]:
INPUT_COLS = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]

inputs = {
    colname: Input(name=colname, shape=(1,), dtype="float32")
    for colname in INPUT_COLS
}

### 前処理ロジックの定義

次に、特徴量の前処理を定義します。
1.  **バケット化（Bucketization）:** `Discretization` レイヤーは、あらかじめ定義されたバケット（`latbuckets`, `lonbuckets`）に割り当てることで、連続値の緯度/経度の入力をカテゴリカル特徴量に変換します。
2.  **特徴量の交差（Feature Crossing）:** `HashedCrossing` レイヤーは、これらのバケット化された特徴量を結合して相互作用特徴量（乗車地点、降車地点、乗車-降車の相互作用など）を作成します。これはモデルのワイド（Wide）部分に有用です。

Functional API を使用すると、各接続を個別に定義できるため、より複雑なモデル アーキテクチャが可能になります。

**演習:** 以下のコードを完成させ、バケット化（`Discretization` レイヤー）と特徴量の交差（`HashedCrossing` レイヤー）を組み込む前処理レイヤーを定義してください。

In [ ]:
dnn_hidden_units = [32, 8]
NBUCKETS = 16

# バケット化の境界を定義
latbuckets = np.linspace(start=40.5, stop=41.0, num=NBUCKETS).tolist()
lonbuckets = np.linspace(start=-74.2, stop=-73.7, num=NBUCKETS).tolist()

# Discretization レイヤーによるバケット化
plon = Discretization(lonbuckets, name="plon_bkt")(inputs["pickup_longitude"])
plat = # TODO
dlon = # TODO
dlat = # TODO

# HashedCrossing レイヤーによる特徴量の交差
p_fc = HashedCrossing(num_bins=(NBUCKETS + 1) ** 2, name="p_fc")((plon, plat))
d_fc = # TODO
pd_fc = # TODO

### Deep パスの構築

Deep パスは汎化を担います。
1.  `pd_fc` 交差特徴量を `Embedding` レイヤーを使用して埋め込み、密ベクトル表現を作成します。
2.  これらの埋め込みは、元の数値入力座標と連結されます。
3.  その結果が、ReLU 活性化関数を備えた一連の `Dense` レイヤーに渡されます。

**演習:** `Embedding` レイヤーの定義を完成させてディープ モデルを構築してください。

In [ ]:
# Embedding レイヤーを使用して `pd_cross` の埋め込みを作成
pd_embed = # TODO

# 連結してディープ ネットワークの入力を定義
deep = Concatenate(name="deep_input")(
    [
        inputs["pickup_longitude"],
        inputs["pickup_latitude"],
        inputs["dropoff_longitude"],
        inputs["dropoff_latitude"],
        Flatten(name="flatten_embedding")(pd_embed),
    ]
)

# 隠れ Dense レイヤーを追加
for i, num_nodes in enumerate(dnn_hidden_units, start=1):
    deep = Dense(num_nodes, activation="relu", name=f"hidden_{i}")(deep)

### Wide パスの構築

Wide パスは記憶を担います。
1.  交差特徴量（`p_fc`、`d_fc`、`pd_fc`）は、`CategoryEncoding` レイヤーを使用してワンホット（one-hot）エンコードされます。
2.  これらのワンホット エンコードされた特徴量が連結され、モデルのワイド部分の入力が形成されます。

**演習:** `CategoryEncoding` レイヤーの定義を完成させてワイド モデルを構築してください。

In [ ]:
# CategoryEncoding レイヤーによるワンホット エンコーディング
p_onehot = CategoryEncoding(num_tokens=(NBUCKETS + 1) ** 2, name="p_onehot")(
    p_fc
)
d_onehot = # TODO
pd_onehot = # TODO

# 連結してワイド ネットワークの入力を定義
wide = Concatenate(name="wide_input")() # TODO

### Wide パスと Deep パスの結合

`deep` パスと `wide` パスの出力が連結されます。この結合された Tensor は、予測を生成するために 1 ユニット（回帰のため活性化関数なし）の最終 `Dense` レイヤーに入力されます。

**演習:** ワイド モデルとディープ モデルの結合を定義し、最終的な出力レイヤーを定義してください。

In [ ]:
# ワイド ネットワークとディープ ネットワークの結合
concat = # TODO

# 最終出力レイヤーを定義
output = # TODO

### モデルのインスタンス化とコンパイル

Functional API では、`keras.Model` を使用して、`inputs`（`Input` レイヤーの辞書）と `outputs`（最終的な `Dense` レイヤー）を指定することでモデルを定義します。

In [ ]:
model = Model(inputs=list(inputs.values()), outputs=output)

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[RootMeanSquaredError()],
)

Functional API では、`keras.utils.plot_model` を使用してモデル構造の図を生成できます。

In [ ]:
keras.utils.plot_model(model, show_layer_names=True, rankdir="LR")

## Wide & Deep モデルのトレーニング
次に、トレーニング変数を設定し、トレーニング用と検証用のデータセットを作成して、モデルをトレーニングします。

（トレーニングを `NUM_TRAIN_EXAMPLES` と `NUM_EVALS` で表現する理由、およびこのトレーニング コードにおいてエポック数が実際に実行する評価の回数と等しくなる理由の詳細については、ブログ記事 [ML Design Pattern #3: Virtual Epochs](https://medium.com/google-cloud/ml-design-pattern-3-virtual-epochs-f842296de730) を参照してください。）

In [ ]:
BATCH_SIZE = 64
NUM_TRAIN_EXAMPLES = 10000 * 10  # トレーニング データセットはリピート（ループ）します
NUM_EVALS = 10  # 評価を行う回数
NUM_EVAL_EXAMPLES = 1000  # 妥当なサンプルを得るのに十分な数

trainds = create_dataset(
    pattern="../data/taxi-train.csv", batch_size=BATCH_SIZE, mode="train"
)

evalds = create_dataset(
    pattern="../data/taxi-valid.csv", batch_size=BATCH_SIZE, mode="eval"
).take(NUM_EVAL_EXAMPLES // BATCH_SIZE)

In [ ]:
%%time
steps_per_epoch = NUM_TRAIN_EXAMPLES // (BATCH_SIZE * NUM_EVALS)

OUTDIR = "./taxi_trained"
shutil.rmtree(path=OUTDIR, ignore_errors=True)  # 毎回新しく開始

history = model.fit(
    x=trainds,
    steps_per_epoch=steps_per_epoch,
    epochs=NUM_EVALS,
    validation_data=evalds,
    callbacks=[TensorBoard(OUTDIR)],
)

前と同様に、履歴（history）を調べることで、トレーニング セットと検証セットに対するトレーニングの進行に伴う RMSE の変化を確認できます。

In [ ]:
RMSE_COLS = ["root_mean_squared_error", "val_root_mean_squared_error"]

pd.DataFrame(history.history)[RMSE_COLS].plot()

Copyright 2025 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.